# Designing a Feature Table Schema for Text and LLM Embeddings

## 1. Designing the schema

## Embeddings
- Reviews that mean similar things end up close together, even with different words (`okay`, `decent` ≈ `average`).
- Closeness is measured with cosine similarity → used for clustering, semantic search and ML features.

**Why use them in addition to TF-IDF?**
- TF-IDF → matches exact words only: `good` ≠ `great`, word order and negation (`not good`) are lost.
- Embeddings → capture meaning and context, in a compact fixed-size vector (no thousands of zeros).
- TF-IDF stays useful → interpretable keywords (`spilled`, `late`), cheap, computed from our own data.
- Together → the model gets both precise keywords (TF-IDF) and overall meaning (embeddings).


### Testing Access to a Hosted Embedding Model

In [0]:
%sql
SELECT ai_query(
  'databricks-gte-large-en',
  'This is a test sentence for embeddings.'
) AS test_embedding;

### Checking the Embedding Dimension

In [0]:
%sql
SELECT size(test_embedding) AS embedding_dimension
FROM (
  SELECT ai_query(
    'databricks-gte-large-en',
    'This is a test sentence for embeddings.'
  ) AS test_embedding
);

### Designing the Embedding Feature Table Schema

In [0]:
%sql
CREATE TABLE IF NOT EXISTS pipeline_catalog.tabular_pipeline.customer_review_embeddings (
    customer_id           INT       NOT NULL,
    review_text           STRING,
    embedding             ARRAY<DOUBLE>,
    embedding_model       STRING,
    embedding_dimension   INT,
    generated_at           TIMESTAMP
)
USING DELTA;

### Adding a Primary Key So This Becomes a Real Feature Table

In [0]:
%sql
ALTER TABLE pipeline_catalog.tabular_pipeline.customer_review_embeddings
ADD CONSTRAINT customer_review_embeddings_pk PRIMARY KEY (customer_id);

### Adding a Constraint to Catch Dimension Mismatches

In [0]:
%sql
ALTER TABLE pipeline_catalog.tabular_pipeline.customer_review_embeddings
ADD CONSTRAINT correct_embedding_dimension
CHECK (embedding_dimension = 1024);

### Adding a Descriptive Comment

In [0]:
%sql
COMMENT ON TABLE pipeline_catalog.tabular_pipeline.customer_review_embeddings IS
'Stores LLM-generated embeddings for customer reviews. Embeddings produced by databricks-gte-large-en, 1024 dimensions. Embedding model and dimension are tracked per row to detect model drift. Primary key: customer_id.';

### Verifying the Schema

In [0]:
%sql
DESCRIBE TABLE EXTENDED pipeline_catalog.tabular_pipeline.customer_review_embeddings;